# 1.04 — Strings — solutions

Lesson: [`lessons/1.04_strings.md`](../lessons/1.04_strings.md)

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — initials

`initials(full_name)` → `"A.L."` for `"ada lovelace"`. Works for any number of names; ignore extra spaces.

In [ ]:
def initials(full_name):
    parts = full_name.split()
    result = ""
    for p in parts:
        result += p[0].upper() + "."
    return result

In [ ]:
check(initials, [(("ada lovelace",), "A.L."), (("  kasun  perera ",), "K.P."), (("Grace Brewster Hopper",), "G.B.H.")])

## Problem 2 — mask a card number

`mask_card(number)` keeps the last 4 characters and replaces everything before them with `*`.

In [ ]:
def mask_card(number):
    return "*" * (len(number) - 4) + number[-4:]

In [ ]:
check(mask_card, [(("4111111111111111",), "************1111"), (("12345",), "*2345"), (("9876",), "9876")])

## Problem 3 — palindromes

`is_palindrome(text)` ignores case and spaces: `"Never odd or even"` → `True`.

In [ ]:
def is_palindrome(text):
    cleaned = text.replace(" ", "").lower()
    return cleaned == cleaned[::-1]

In [ ]:
check(is_palindrome, [(("Never odd or even",), True), (("racecar",), True), (("python",), False), (("",), True)])

## Problem 4 — domain from a URL

`extract_domain(url)` returns the domain without `http://` / `https://`, without a leading `www.`, and without any path.

In [ ]:
def extract_domain(url):
    if "://" in url:
        url = url.split("://", 1)[1]
    host = url.split("/")[0]
    if host.startswith("www."):
        host = host[4:]
    return host

In [ ]:
check(extract_domain, [
    (("https://www.example.com/path/page",), "example.com"),
    (("http://data.gov.lk",), "data.gov.lk"),
    (("www.cbsl.gov.lk/en/rates",), "cbsl.gov.lk"),
])

## Problem 5 — title case by hand

`title_case(text)` capitalises the first letter of each word and lower-cases the rest, **without** using `.title()` or `.capitalize()`. Words are separated by single spaces.

In [ ]:
def title_case(text):
    words = text.split(" ")
    fixed = []
    for w in words:
        if w:
            fixed.append(w[0].upper() + w[1:].lower())
        else:
            fixed.append(w)
    return " ".join(fixed)

In [ ]:
check(title_case, [(("hello world",), "Hello World"), (("dATA sCIENCE",), "Data Science"), (("a",), "A")])

## Problem 6 — clean an amount

`clean_amount(text)` turns strings like `"LKR 1,250.50"`, `"Rs. 900"` or `" 12,000 "` into a `float`. Remove the currency prefixes `LKR` and `Rs.`, commas and spaces.

In [ ]:
def clean_amount(text):
    cleaned = text.replace("LKR", "").replace("Rs.", "").replace(",", "").strip()
    return float(cleaned)

In [ ]:
check(clean_amount, [(("LKR 1,250.50",), 1250.5), (("Rs. 900",), 900.0), ((" 12,000 ",), 12000.0)])

## Problem 7 ★ — snake_case to camelCase

`snake_to_camel("monthly_total_spend")` → `"monthlyTotalSpend"`.

In [ ]:
def snake_to_camel(name):
    parts = name.split("_")
    result = parts[0].lower()
    for p in parts[1:]:
        if p:
            result += p[0].upper() + p[1:].lower()
    return result

In [ ]:
check(snake_to_camel, [(("monthly_total_spend",), "monthlyTotalSpend"), (("id",), "id"), (("user_ID",), "userId")])

## Problem 8 ★ — validate an NIC number

A Sri Lankan NIC number is valid if **either**:

- it's 10 characters: 9 digits followed by `V` or `X` (either case), **or**
- it's 12 characters, all digits.

Write `is_valid_nic(text)`. Ignore surrounding spaces.

In [ ]:
def is_valid_nic(text):
    t = text.strip()
    if len(t) == 10:
        return t[:9].isdigit() and t[9] in "VvXx"
    if len(t) == 12:
        return t.isdigit()
    return False

In [ ]:
check(is_valid_nic, [
    (("912345678V",), True), ((" 912345678x ",), True), (("199112345678",), True),
    (("91234567V",), False), (("91234567AV",), False), (("19911234567X",), False), (("",), False),
])